# 6 — Build core edges and evidence

Purpose: make the canonical relation/evidence build path reproducible without forking pipeline logic into notebook-local code. The notebook calls `manage_db` builders/audits directly, uses DuckDB only for display/validation queries, and defaults to sample/read-only mode.

Policy: relation names follow source-native endpoint/assertion semantics. Gene-level rows stay in gene relations; protein relations require direct protein/isoform evidence, not gene→protein projection.

In [ ]:
from __future__ import annotations
import json, os, sys
from dataclasses import asdict, is_dataclass
from pathlib import Path
import duckdb
import pandas as pd
import pyarrow.parquet as pq
try:
    from IPython import get_ipython
    if get_ipython() is None:
        raise RuntimeError("not running inside IPython")
    from IPython.display import display
except Exception:  # pragma: no cover - plain Python fallback for lightweight validation
    def display(obj):
        print(obj)

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "manage_db").exists():
    REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

SAMPLE_MODE = os.environ.get("JOUVENCE_NOTEBOOK_SAMPLE_MODE", os.environ.get("TXGNN_NOTEBOOK_SAMPLE_MODE", "1")) != "0"
RUN_BUILD = os.environ.get("JOUVENCE_NOTEBOOK_RUN_BUILD", os.environ.get("TXGNN_NOTEBOOK_RUN_BUILD", "0")) == "1"
RUN_FULL_VALIDATION = os.environ.get("JOUVENCE_NOTEBOOK_FULL_VALIDATION", os.environ.get("TXGNN_NOTEBOOK_FULL_VALIDATION", "0")) == "1"
FUSE_KG_ROOT = Path("/Users/jkobject/mnt/gcs/jouvencekb-kg/v2")
def path_exists(path: Path) -> bool:
    try:
        return path.exists()
    except OSError:
        return False
DEFAULT_KG_ROOT = Path(os.environ.get("JOUVENCE_KG_ROOT", os.environ.get("TXGNN_KG_ROOT", str(FUSE_KG_ROOT if path_exists(FUSE_KG_ROOT) else REPO_ROOT / "data" / "kg"))))
DATA_DIR = Path(os.environ.get("JOUVENCE_DATA_DIR", os.environ.get("TXGNN_DATA_DIR", str(REPO_ROOT / "data"))))
OT_DIR = DATA_DIR / "opentargets"
BUILD_KG_ROOT = DATA_DIR / "kg"
print(f"repo={REPO_ROOT}")
print(f"sample_mode={SAMPLE_MODE} run_build={RUN_BUILD} run_full_validation={RUN_FULL_VALIDATION}")
print(f"default_kg_root={DEFAULT_KG_ROOT}")
print(f"default_kg_root_exists={path_exists(DEFAULT_KG_ROOT)} data_dir={DATA_DIR}")


In [ ]:
from manage_db import kg_storage
from manage_db.audit_edge_evidence import audit_edge_evidence
from manage_db.audit_kg_coverage import audit_coverage
from manage_db.validate_kg import validate_duckdb, validate_streaming
from manage_db.ingest_opentargets import (
    ingest_targets, ingest_orthology, ingest_go, ingest_biosample,
    ingest_interactions, ingest_evidence, ingest_indication, ingest_mechanism_of_action,
    ingest_target_essentiality, ingest_disease_phenotype, ingest_expression,
    ingest_pharmacogenomics, ingest_variant_protein_changes,
    ingest_evidence_backed_variants, ingest_enhancers,
)
BUILDER_GROUPS = {
    "structural_edges": [ingest_targets, ingest_orthology, ingest_go, ingest_biosample],
    "source_backed_edges_and_evidence": [ingest_interactions, ingest_evidence, ingest_indication, ingest_mechanism_of_action, ingest_target_essentiality, ingest_disease_phenotype, ingest_expression, ingest_pharmacogenomics, ingest_variant_protein_changes, ingest_evidence_backed_variants, ingest_enhancers],
}
for group, functions in BUILDER_GROUPS.items():
    print(group)
    for fn in functions:
        assert callable(fn), fn
        print("  -", fn.__name__)

In [ ]:
def parquet_path(kind: str, name: str, root: Path = DEFAULT_KG_ROOT) -> Path:
    return root / kind / f"{name}.parquet"

def parquet_exists(kind: str, name: str, root: Path = DEFAULT_KG_ROOT) -> bool:
    return parquet_path(kind, name, root).exists()

def parquet_rows(path: Path) -> int | None:
    if not path.exists():
        return None
    return pq.ParquetFile(path).metadata.num_rows

def relation_inventory(relations: list[str], root: Path = DEFAULT_KG_ROOT) -> pd.DataFrame:
    rows = []
    for relation in relations:
        edge = parquet_path("edges", relation, root)
        evidence = parquet_path("evidence", relation, root)
        rows.append({
            "relation": relation,
            "edge_file": edge.exists(),
            "edge_rows": parquet_rows(edge),
            "evidence_file": evidence.exists(),
            "evidence_rows": parquet_rows(evidence),
        })
    return pd.DataFrame(rows)

def duckdb_relation_summary(relation: str, root: Path = DEFAULT_KG_ROOT) -> pd.DataFrame:
    evidence_path = parquet_path("evidence", relation, root)
    if not evidence_path.exists():
        return pd.DataFrame([{"relation": relation, "status": "missing evidence parquet", "path": str(evidence_path)}])
    con = duckdb.connect(":memory:")
    try:
        cols = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{evidence_path.as_posix()}')").df()["column_name"].tolist()
        dimensions = [c for c in ["source", "source_dataset", "evidence_type", "predicate", "direction"] if c in cols]
        if not dimensions:
            return pd.DataFrame([{"relation": relation, "status": "no standard provenance columns", "path": str(evidence_path)}])
        select_dims = ", ".join(dimensions)
        query = f"""
            SELECT {select_dims}, count(*) AS rows
            FROM read_parquet('{evidence_path.as_posix()}')
            GROUP BY {select_dims}
            ORDER BY rows DESC
            LIMIT 50
        """
        return con.execute(query).df()
    finally:
        con.close()

def edge_evidence_key_anti_join(relation: str, root: Path = DEFAULT_KG_ROOT) -> pd.DataFrame:
    edge_path = parquet_path("edges", relation, root)
    evidence_path = parquet_path("evidence", relation, root)
    if not edge_path.exists() or not evidence_path.exists():
        return pd.DataFrame([{"relation": relation, "status": "missing edge or evidence parquet", "edge_path": str(edge_path), "evidence_path": str(evidence_path)}])
    con = duckdb.connect(":memory:")
    try:
        query = f"""
        WITH edge_keys AS (
            SELECT relation || '|' || x_id || '|' || y_id AS edge_key
            FROM read_parquet('{edge_path.as_posix()}')
        ), evidence_keys AS (
            SELECT COALESCE(edge_key, relation || '|' || x_id || '|' || y_id) AS edge_key
            FROM read_parquet('{evidence_path.as_posix()}')
        )
        SELECT
          (SELECT count(*) FROM edge_keys) AS edge_rows,
          (SELECT count(*) FROM evidence_keys) AS evidence_rows,
          (SELECT count(*) FROM edge_keys e WHERE NOT EXISTS (SELECT 1 FROM evidence_keys v WHERE v.edge_key = e.edge_key)) AS edges_without_evidence,
          (SELECT count(*) FROM evidence_keys v WHERE NOT EXISTS (SELECT 1 FROM edge_keys e WHERE e.edge_key = v.edge_key)) AS evidence_without_edge
        """
        return con.execute(query).df()
    finally:
        con.close()

def endpoint_type_summary(relation: str, root: Path = DEFAULT_KG_ROOT) -> pd.DataFrame:
    edge_path = parquet_path("edges", relation, root)
    if not edge_path.exists():
        return pd.DataFrame([{"relation": relation, "status": "missing edge parquet", "path": str(edge_path)}])
    con = duckdb.connect(":memory:")
    try:
        return con.execute(f"""
            SELECT x_type, y_type, count(*) AS rows
            FROM read_parquet('{edge_path.as_posix()}')
            GROUP BY 1, 2
            ORDER BY rows DESC
        """).df()
    finally:
        con.close()

## Guarded production build skeleton

This cell calls existing builders directly. It is off by default because it writes large Parquet outputs.

In [ ]:
if RUN_BUILD:
    out_dir = BUILD_KG_ROOT
    out_dir.mkdir(parents=True, exist_ok=True)
    kg_root = kg_storage.open_kg_root(str(out_dir))
    structural_results = {
        "ingest_targets": ingest_targets(OT_DIR, out_dir, kg_root),
        "ingest_orthology": ingest_orthology(OT_DIR, out_dir, kg_root),
        "ingest_go": ingest_go(OT_DIR, out_dir, kg_root),
        "ingest_biosample": ingest_biosample(OT_DIR, out_dir, kg_root),
    }
    evidence_results = {
        "ingest_interactions": ingest_interactions(OT_DIR, out_dir, kg_root),
        "ingest_evidence": ingest_evidence(OT_DIR, out_dir, kg_root),
        "ingest_indication": ingest_indication(OT_DIR, out_dir, kg_root),
        "ingest_mechanism_of_action": ingest_mechanism_of_action(OT_DIR, out_dir, kg_root),
        "ingest_target_essentiality": ingest_target_essentiality(OT_DIR, out_dir, kg_root),
        "ingest_disease_phenotype": ingest_disease_phenotype(OT_DIR, out_dir, kg_root),
        "ingest_expression": ingest_expression(OT_DIR, out_dir, kg_root),
        "ingest_pharmacogenomics": ingest_pharmacogenomics(OT_DIR, out_dir, kg_root),
        "ingest_variant_protein_changes": ingest_variant_protein_changes(OT_DIR, out_dir, kg_root),
        "ingest_evidence_backed_variants": ingest_evidence_backed_variants(OT_DIR, out_dir, kg_root),
        "ingest_enhancers": ingest_enhancers(OT_DIR, out_dir, kg_root),
    }
    display(pd.DataFrame([{"stage":"structural", "function":k, "result":str(v)} for k,v in structural_results.items()] + [{"stage":"evidence", "function":k, "result":str(v)} for k,v in evidence_results.items()]))
else:
    display(pd.DataFrame([{"stage": stage, "function": fn.__name__, "status": "skipped; set JOUVENCE_NOTEBOOK_RUN_BUILD=1"} for stage, fns in BUILDER_GROUPS.items() for fn in fns]))

## Cached relation/evidence inventory and provenance summaries

In [ ]:
CORE_RELATIONS = ["gene_has_transcript", "transcript_encodes_protein", "gene_ortholog_gene", "pathway_contains_gene", "gene_interacts_gene", "disease_associated_gene", "molecule_targets_gene", "molecule_treats_disease", "mutation_associated_disease", "enhancer_regulates_gene", "tissue_expresses_gene"]
display(relation_inventory(CORE_RELATIONS))
for relation in ["gene_interacts_gene", "molecule_targets_gene", "pathway_contains_gene"]:
    print(f"\n=== {relation} evidence provenance ===")
    display(duckdb_relation_summary(relation))

## Evidence support and endpoint validation gates

In [ ]:
relations_to_audit = [r for r in ["gene_interacts_gene", "molecule_targets_gene"] if parquet_exists("edges", r) or parquet_exists("evidence", r)]
if relations_to_audit:
    try:
        audit = audit_edge_evidence(DEFAULT_KG_ROOT, relations=relations_to_audit)
        display(pd.DataFrame([asdict(report) | {"ok": report.ok} for report in audit.relation_reports.values()]))
    except Exception as exc:
        print(f"audit_edge_evidence failed in this environment: {exc}")
    for relation in relations_to_audit:
        print(f"\nDuckDB edge/evidence anti-join: {relation}")
        display(edge_evidence_key_anti_join(relation))
        print("Endpoint type summary")
        display(endpoint_type_summary(relation))
else:
    print("No cached edge/evidence relations available for the support audit demo.")

if RUN_FULL_VALIDATION:
    report = validate_duckdb(DEFAULT_KG_ROOT, threads=2, temp_dir=REPO_ROOT / "artifacts" / "cache" / "duckdb-tmp")
    assert report.total_dangling_edges == 0
else:
    print("Full DuckDB validation skipped; set JOUVENCE_NOTEBOOK_FULL_VALIDATION=1 on a local/mounted KG root.")

if DEFAULT_KG_ROOT.exists():
    coverage = audit_coverage(DEFAULT_KG_ROOT)
    print(f"nodes={len(coverage.node_counts)} total_nodes={coverage.total_nodes:,}")
    print(f"edges={len(coverage.edge_counts)} total_edges={coverage.total_edges:,}")